# OCR con IA desde Google Colab

OCR de PDFs con PaddleOCR-VL-1.5. Entorno de ejecucion -> GPU (T4). Corre las celdas en orden.

## 1. Configuracion

`USAR_DRIVE = False`: todo en la maquina virtual, se pierde al reiniciar.
`USAR_DRIVE = True`: guarda paquetes y modelo en tu Drive y la proxima vez arranca en ~1 minuto.
`FORMATO = 'md'`: Markdown puro, tablas de pipes. `'crudo'`: tal cual lo devuelve el modelo, con tablas en HTML.

In [ ]:
USAR_DRIVE = False
CARPETA_DRIVE = '/content/drive/MyDrive/ocr_ia'
PAGINAS_MAX = 5
FORMATO = 'md'


## 2. Entorno

In [ ]:
import os, subprocess, sys

DIR_PAQUETES = DIR_CACHE = None

if USAR_DRIVE:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    DIR_PAQUETES = f'{CARPETA_DRIVE}/paquetes'
    DIR_CACHE = f'{CARPETA_DRIVE}/pip_cache'
    hf, px = f'{CARPETA_DRIVE}/huggingface', f'{CARPETA_DRIVE}/paddlex'
    for d in (DIR_PAQUETES, DIR_CACHE, hf, px):
        os.makedirs(d, exist_ok=True)
    os.environ['HF_HOME'] = hf
    destino = os.path.expanduser('~/.paddlex')
    if os.path.islink(destino):
        os.unlink(destino)
    elif os.path.isdir(destino):
        import shutil
        shutil.rmtree(destino)
    os.symlink(px, destino)

subprocess.run('apt-get -qq install -y poppler-utils', shell=True,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

PAQUETES = [
    ['paddlepaddle-gpu==3.2.1', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'],
    ['paddleocr[doc-parser]>=3.4.0'],
    ['pdf2image', 'pypdf'],
]

def instalar(args):
    base = [sys.executable, '-m', 'pip', 'install', '-q', '--timeout', '180', '--retries', '10']
    if DIR_CACHE:
        base += ['--cache-dir', DIR_CACHE]
    if DIR_PAQUETES:
        base += ['--target', DIR_PAQUETES, '--upgrade']
    for intento in range(3):
        if subprocess.run(base + args).returncode == 0:
            return True
    return False

if DIR_PAQUETES and os.path.exists(f'{DIR_PAQUETES}/.ok'):
    print('Paquetes ya instalados en Drive.')
else:
    for p in PAQUETES:
        print('instalando', p[0])
        if not instalar(p) and 'paddlepaddle-gpu' in p[0]:
            print('  GPU no disponible, usando Paddle CPU')
            instalar(['paddlepaddle>=3.0.0'])
    if DIR_PAQUETES:
        open(f'{DIR_PAQUETES}/.ok', 'w').close()

if DIR_PAQUETES and DIR_PAQUETES not in sys.path:
    sys.path.append(DIR_PAQUETES)

print('Entorno listo.')


## 3. Modelo

In [ ]:
from paddleocr import PaddleOCRVL

pipeline = PaddleOCRVL(pipeline_version='v1.5')
print('Modelo cargado.')


## 4. Subir PDFs

Podes elegir varios a la vez con Ctrl+click.

In [ ]:
import os
from google.colab import files

os.makedirs('/content/pdfs', exist_ok=True)
RUTAS = []
for nombre, datos in files.upload().items():
    ruta = f'/content/pdfs/{nombre}'
    open(ruta, 'wb').write(datos)
    RUTAS.append(ruta)

print(f'{len(RUTAS)} PDF(s) cargados.')


## 5. OCR

In [ ]:
import os, time
from pypdf import PdfReader, PdfWriter

SALIDA = '/content/salida'
os.makedirs(SALIDA, exist_ok=True)
resultados = {}

def recortar(ruta):
    if not PAGINAS_MAX:
        return ruta
    lector = PdfReader(ruta)
    if len(lector.pages) <= PAGINAS_MAX:
        return ruta
    escritor = PdfWriter()
    for pag in lector.pages[:PAGINAS_MAX]:
        escritor.add_page(pag)
    corto = ruta.replace('.pdf', '_corto.pdf')
    with open(corto, 'wb') as f:
        escritor.write(f)
    return corto

for i, ruta in enumerate(RUTAS, 1):
    nombre = os.path.basename(ruta)
    print(f'[{i}/{len(RUTAS)}] {nombre}')
    try:
        carpeta = f'{SALIDA}/{os.path.splitext(nombre)[0]}'
        os.makedirs(carpeta, exist_ok=True)
        t0 = time.time()
        for res in pipeline.predict(recortar(ruta)):
            res.save_to_markdown(save_path=carpeta)
            res.save_to_json(save_path=carpeta)
        resultados[ruta] = carpeta
        print(f'      ok en {time.time()-t0:.1f}s')
    except Exception as e:
        print(f'      fallo: {type(e).__name__}: {e}')

print(f'\n{len(resultados)} de {len(RUTAS)} procesados.')


## 6. Formato de salida

`FORMATO = 'md'` pasa las tablas HTML a tablas de pipes y saca el HTML suelto.
`'crudo'` deja la salida del modelo sin tocar.

In [ ]:
import glob, html, os, re

RE_TABLA = re.compile(r'<table\b.*?</table>', re.S | re.I)
RE_FILA = re.compile(r'<tr\b[^>]*>(.*?)</tr>', re.S | re.I)
RE_CELDA = re.compile(r'<(t[hd])\b([^>]*)>(.*?)</\1>', re.S | re.I)
RE_COLSPAN = re.compile(r'colspan\s*=\s*["\']?(\d+)', re.I)
RE_IMAGEN = re.compile(r'!\[([^\]]*)\]\(([^)]+)\)')
RE_BR = re.compile(r'<br\s*/?>', re.I)
RE_SOBRAS = re.compile(
    r'</?(div|span|p|font|b|i|u|em|strong|sup|sub|center|a)\b[^>]*>', re.I)

def limpiar(bruto):
    t = re.sub(r'<[^>]+>', '', RE_BR.sub(' ', bruto))
    return ' '.join(html.unescape(t).split()).replace('|', r'\|')

def celdas(fila):
    salida = []
    for _, atributos, contenido in RE_CELDA.findall(fila):
        ancho = RE_COLSPAN.search(atributos)
        salida.append(limpiar(contenido))
        salida += [''] * (int(ancho.group(1)) - 1 if ancho else 0)
    return salida

def tabla_a_pipes(bloque):
    filas = [c for c in (celdas(f) for f in RE_FILA.findall(bloque)) if c]
    if not filas:
        return ''
    ancho = max(len(f) for f in filas)
    filas = [f + [''] * (ancho - len(f)) for f in filas]
    if not any(c.strip() for c in filas[0]):
        filas[0] = [f'col {i + 1}' for i in range(ancho)]
    lineas = ['| ' + ' | '.join(filas[0]) + ' |', '|' + ' --- |' * ancho]
    return '\n'.join(lineas + ['| ' + ' | '.join(f) + ' |' for f in filas[1:]])

def a_markdown(texto):
    texto = RE_TABLA.sub(lambda m: '\n' + tabla_a_pipes(m.group(0)) + '\n', texto)
    texto = RE_SOBRAS.sub('', RE_BR.sub('\n', texto))
    texto = html.unescape(texto)
    return re.sub(r'\n{3,}', '\n\n', re.sub(r'[ \t]+\n', '\n', texto)).strip()

textos = {}
for ruta, carpeta in resultados.items():
    piezas = sorted(glob.glob(f'{carpeta}/**/*.md', recursive=True))
    partes = []
    for pieza in piezas:
        trozo = open(pieza, encoding='utf-8').read().strip()
        base = os.path.dirname(pieza)
        trozo = RE_IMAGEN.sub(
            lambda m: f'![{m.group(1)}]({os.path.join(base, m.group(2))})'
            if not m.group(2).startswith(('http', 'data:', '/')) else m.group(0), trozo)
        partes.append(a_markdown(trozo) if FORMATO == 'md' else trozo)
    textos[ruta] = '\n\n'.join(partes)
    filas = sum(1 for l in textos[ruta].splitlines() if l.lstrip().startswith('|'))
    print(f'{os.path.basename(ruta)}: {len(textos[ruta])} chars, {filas} lineas de tabla')


## 7. Ver el resultado

In [ ]:
import os
from IPython.display import Markdown, display

for ruta, texto in textos.items():
    print('=' * 60)
    print(os.path.basename(ruta))
    display(Markdown(texto[:5000]))


## 8. Guardar y descargar

Un `.md` por PDF. Con varios baja un zip.

In [ ]:
import datetime, os, shutil
from google.colab import files

sello = datetime.datetime.now().strftime('%Y%m%d_%H%M')
generados = []

for ruta, texto in textos.items():
    base = os.path.splitext(os.path.basename(ruta))[0]
    cabecera = f'# OCR de {os.path.basename(ruta)}\n\n- Modelo: PaddleOCR-VL-1.5\n'
    cabecera += f'- Formato: {FORMATO}\n- Fecha: {sello}\n\n---\n\n'
    destino = f'{SALIDA}/{base}_ocr.md'
    with open(destino, 'w', encoding='utf-8') as f:
        f.write(cabecera + texto)
    generados.append(destino)
    print(os.path.basename(destino))

if USAR_DRIVE and generados:
    carpeta_md = f'{CARPETA_DRIVE}/resultados/{sello}'
    os.makedirs(carpeta_md, exist_ok=True)
    for g in generados:
        shutil.copy(g, carpeta_md)
    print('Copiados a', carpeta_md)

if len(generados) == 1:
    files.download(generados[0])
elif generados:
    bolsa = f'/content/ocr_{sello}'
    os.makedirs(bolsa, exist_ok=True)
    for g in generados:
        shutil.copy(g, bolsa)
    shutil.make_archive(bolsa, 'zip', bolsa)
    files.download(bolsa + '.zip')
